# FlyWire 視覚系における側抑制 (lateral inhibition) の検証

ショウジョウバエ視覚系では「至るところで側抑制が起きている」とよく言われる。FlyWire の連結体だけからこの主張がどこまで支持できるかを以下の問いで確認する。

1. **Q1** 視覚系の全シナプスのうち、抑制性 (GABA / GLUT / HIS) はどれくらいの割合か?
2. **Q2** 抑制性出力をメインに持つ cell type はどれくらい広範に存在するか?
3. **Q3** 同タイプ間 (within-type) の抑制接続 — 最も直接的な側抑制シグナル — はどの程度起きているか?
4. **Q4** wide-field な抑制 (= 1 ニューロンの出力シナプスが広い範囲に分布) は本当に抑制 dominant cell type で顕著か?
5. **Q5** 既知の側抑制回路 (Lamina の Lai, Medulla の Dm 系) が実際にデータで再現されるか?

**抑制性 nt の定義** Drosophila では GABA (GABA-A Cl-)、HIS (HCl1, R1-6 用)、GLUT (GluCl 経由でしばしば抑制性) を抑制性とみなす (`{GABA, GLUT, HIS}`)。`{ACH}` を興奮性。

**caveat** `nt_type` の多くは ML 推定。R1-6 のみドメイン補正で HIS。個々の予測には誤りがあるので、aggregate を見る。

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "src").is_dir() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_DIR
from src.data import FlyWireDataManager

INHIBITORY_NT = {"GABA", "GLUT", "HIS"}
EXCITATORY_NT = {"ACH"}

m = FlyWireDataManager()
neurons = m.optic_lobe_neurons_df.copy()
conn = m.optic_lobe_connections_df.copy()

def classify_nt(x):
    if x in INHIBITORY_NT:
        return "inh"
    if x in EXCITATORY_NT:
        return "exc"
    return "other"

conn["sign"] = conn["nt_type"].map(classify_nt)
conn["same_type"] = conn["pre_primary_type"] == conn["post_primary_type"]
print(f"neurons={len(neurons):,}, edges={len(conn):,}")
print(f"sign distribution by edges: {conn['sign'].value_counts().to_dict()}")

## Q1. 全シナプスの I/E バランス

視覚系全体で抑制性シナプスがどの程度を占めるか。30〜40% を超えるようなら抑制が「至るところ」にあると言える。

In [ ]:
syn_by_sign = conn.groupby("sign")["syn_count"].sum()
inh = int(syn_by_sign.get("inh", 0))
exc = int(syn_by_sign.get("exc", 0))
other = int(syn_by_sign.get("other", 0))
total = inh + exc + other
ie_share = inh / (inh + exc)
print(f"Total synapses     : {total:,}")
print(f"  Inhibitory  (GABA/GLUT/HIS): {inh:,} ({inh/total:.1%})")
print(f"  Excitatory  (ACH)          : {exc:,} ({exc/total:.1%})")
print(f"  Other       (DA/SER/OCT)   : {other:,} ({other/total:.1%})")
print(f"  I / (I+E)                   = {ie_share:.1%}")

nt_syn = conn.groupby("nt_type", dropna=False)["syn_count"].sum().sort_values()
colors = ["tab:red" if x in INHIBITORY_NT else ("tab:blue" if x in EXCITATORY_NT else "gray") for x in nt_syn.index]
fig, ax = plt.subplots(figsize=(8, 4))
nt_syn.plot.barh(ax=ax, color=colors)
ax.set(xlabel="# synapses", title="Synapse count by nt_type  (red=inh, blue=exc, gray=other)")
plt.tight_layout()

## Q2. 抑制性出力を持つ cell type の広がり

視覚系の cell type のうち、出力のメインが抑制性であるものがどれくらいあるか。多数あれば「至るところで抑制」に整合。

In [ ]:
type_io = (
    conn.groupby(["pre_primary_type", "sign"])["syn_count"].sum().unstack(fill_value=0)
)
for col in ("inh", "exc", "other"):
    if col not in type_io.columns:
        type_io[col] = 0
type_io["total"] = type_io["inh"] + type_io["exc"] + type_io["other"]
type_io["inh_frac"] = type_io["inh"] / type_io["total"].clip(lower=1)

active = type_io[type_io["total"] >= 1000]
n_active = len(active)
n_pure_exc  = int((active["inh_frac"] < 0.05).sum())
n_mixed     = int(((active["inh_frac"] >= 0.05) & (active["inh_frac"] < 0.5)).sum())
n_mostly_i  = int(((active["inh_frac"] >= 0.5)  & (active["inh_frac"] < 0.95)).sum())
n_pure_inh  = int((active["inh_frac"] >= 0.95).sum())
print(f"Cell types with >=1000 outgoing syns : {n_active}")
print(f"  pure excitatory   (inh<5%)         : {n_pure_exc}")
print(f"  mixed             (5-50%)          : {n_mixed}")
print(f"  mostly inhibitory (50-95%)         : {n_mostly_i}")
print(f"  pure inhibitory   (>=95%)          : {n_pure_inh}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(active["inh_frac"], bins=50, color="steelblue", edgecolor="white")
ax.set(xlabel="inhibitory output fraction (per cell type)", ylabel="# cell types",
       title=f"Distribution of inhibitory output fraction (n={n_active} types, >=1000 syn)")
plt.tight_layout()

print()
print("Top 15 cell types by raw inhibitory output (inh_frac>=50%):")
display(active[active["inh_frac"] >= 0.5].sort_values("inh", ascending=False).head(15)[["inh", "exc", "inh_frac"]])

## Q3. 同タイプ内 (within-type) 抑制

同じ primary_type のニューロン間に抑制接続があると、これは典型的な側抑制 (同じ機能ユニットの隣同士で抑え合う)。
各 cell type について、興奮 / 抑制それぞれの出力のうち何 % が「同じ type」を狙うかを比較する。
**抑制側の自タイプ率 > 興奮側の自タイプ率 (= 対角線より上)** であれば、その type は他より自分の仲間を選択的に抑える傾向がある。

注: 多くの lateral inhibition は専用の抑制ニューロン (Lai, Dm, Pm) 経由で起きるので、within-type だけが lateral inhibition の指標ではないことに注意。

In [ ]:
agg = (
    conn[conn["sign"].isin(["inh", "exc"])]
    .groupby(["pre_primary_type", "sign", "same_type"])["syn_count"].sum()
    .unstack(fill_value=0)
)
agg.columns = ["other_type_syn", "self_type_syn"]
agg["total"] = agg["other_type_syn"] + agg["self_type_syn"]
agg["self_frac"] = agg["self_type_syn"] / agg["total"].clip(lower=1)

wide = agg.reset_index().pivot(index="pre_primary_type", columns="sign", values=["self_frac", "total"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]

viable = wide[(wide["total_inh"].fillna(0) >= 500) & (wide["total_exc"].fillna(0) >= 500)].copy()
n_above = int((viable["self_frac_inh"] > viable["self_frac_exc"]).sum())
print(f"Cell types with >=500 syn in BOTH inh and exc output: {len(viable)}")
print(f"  inh self-targeting > exc self-targeting: {n_above} / {len(viable)} ({n_above/len(viable):.0%})")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(viable["self_frac_exc"], viable["self_frac_inh"], s=18, alpha=0.6, color="tab:purple")
lim = max(viable[["self_frac_inh", "self_frac_exc"]].max().max(), 0.1) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=0.7, alpha=0.5, label="y = x")
ax.set(xlim=(0, lim), ylim=(0, lim),
       xlabel="excitatory self-type syn fraction",
       ylabel="inhibitory self-type syn fraction",
       title="Within-type targeting rate: inh vs exc (each dot = cell type)")
for t, row in viable.iterrows():
    if row["self_frac_inh"] - row["self_frac_exc"] > 0.05:
        ax.annotate(t, (row["self_frac_exc"], row["self_frac_inh"]), fontsize=8, alpha=0.8)
ax.legend()
plt.tight_layout()

print()
print("Top 15 types by (inh_self_frac - exc_self_frac) — clearest within-type lateral inhibition:")
display(viable.assign(diff=viable["self_frac_inh"] - viable["self_frac_exc"])
        .sort_values("diff", ascending=False).head(15)
        [["self_frac_inh", "self_frac_exc", "total_inh", "total_exc", "diff"]])

## Q4. Lateral spread (Δcolumn 単位)

別ノートブック [`column_assignment_validation.ipynb`](column_assignment_validation.ipynb) で `column_assignment.csv` が連結体の retinotopic 構造を強く正確に捉えていることを確認した (Spearman ρ ≈ 0.94, L1→Mi1 は edges の 86% が Δcolumn=0, mean syn が Δ=0 で 30× ピーク)。これにより 3D voxel spread (= stratification と lateral が混在する confound 入りの metric) の代わりに、**Δcolumn 単位の lateral spread** を直接計算できる。

**方法**: 各 pre neuron について
1. 出力先のうち column_assignment に含まれる細胞 (= columnar target) だけに絞る
2. 同じ半球内の edges に限る (column 距離は半球内でしか意味を持たない)
3. post 側の `(p, q)` を syn_count で重み付けして hex centroid を計算
4. 各 target の centroid からの hex 距離を weighted mean
5. cell type 単位で集計し inh vs exc を比較

これで stratification 軸の伸び (Mi1 が M1-M10 を縦断する等) は完全に消え、純粋に「何個眼まで広がっているか」だけが残る。**Dm/Pm/Lai のような wide-field 抑制 interneuron なら大きな Δcolumn spread を持つはず**。

*Pre 側は cell type 任意 (Dm/Pm/Lai 等を含む)。Post 側のみ columnar に限る*

In [ ]:
col_assign = pd.read_csv(
    Path(DATA_DIR) / "raw" / "flywire" / "csv" / "column_assignment.csv",
    dtype={"root_id": str, "column_id": str},
)
col_map = col_assign.set_index("root_id")[["p", "q", "hemisphere"]]
pre_side_map = neurons.drop_duplicates("root_id").set_index("root_id")["side"]

# Edge-level の column 情報
ec = conn[["pre_root_id", "post_root_id", "syn_count", "pre_primary_type", "sign"]].copy()
ec["p_post"]    = ec["post_root_id"].map(col_map["p"])
ec["q_post"]    = ec["post_root_id"].map(col_map["q"])
ec["hemi_post"] = ec["post_root_id"].map(col_map["hemisphere"])
ec["hemi_pre"]  = ec["pre_root_id"].map(pre_side_map)
ec = ec.dropna(subset=["p_post", "q_post", "hemi_pre"])
ec = ec[ec["hemi_pre"] == ec["hemi_post"]]
print(f"Edges where post is columnar AND same hemisphere: {len(ec):,} / {len(conn):,}")

# Per-pre weighted centroid in (p, q)
ec["wp"] = ec["p_post"] * ec["syn_count"]
ec["wq"] = ec["q_post"] * ec["syn_count"]
per_pre = ec.groupby("pre_root_id").agg(
    wp_sum=("wp", "sum"),
    wq_sum=("wq", "sum"),
    w_sum=("syn_count", "sum"),
    n_targets=("syn_count", "size"),
)
per_pre["pc"] = per_pre["wp_sum"] / per_pre["w_sum"]
per_pre["qc"] = per_pre["wq_sum"] / per_pre["w_sum"]
per_pre = per_pre[(per_pre["w_sum"] >= 100) & (per_pre["n_targets"] >= 5)]
print(f"pre neurons with >=100 syn to >=5 columnar targets: {len(per_pre):,}")

# Per-edge: centroid からの hex 距離 (cube projection 風の近似で非整数 centroid に対応)
ec["pc"] = ec["pre_root_id"].map(per_pre["pc"])
ec["qc"] = ec["pre_root_id"].map(per_pre["qc"])
ec = ec.dropna(subset=["pc"])
dp = ec["p_post"] - ec["pc"]
dq = ec["q_post"] - ec["qc"]
ec["d_hex"] = np.sqrt((dp**2 + dq**2 + (dp + dq)**2) / 2)
ec["wd"] = ec["d_hex"] * ec["syn_count"]

# Per-pre weighted-mean spread + unique target columns
spread = ec.groupby("pre_root_id").agg(
    wd_sum=("wd", "sum"),
    w_sum=("syn_count", "sum"),
    n_targets=("syn_count", "size"),
)
spread["spread_wmean"] = spread["wd_sum"] / spread["w_sum"]
# Unique (p, q) target columns
ec["col_tup"] = list(zip(ec["p_post"].astype(int), ec["q_post"].astype(int)))
spread["n_unique_cols"] = ec.groupby("pre_root_id")["col_tup"].nunique()

# Cell type / sign を join
type_map = neurons.drop_duplicates("root_id").set_index("root_id")[["primary_type", "nt_type"]]
spread = spread.join(type_map).dropna(subset=["primary_type"])
spread["sign"] = spread["nt_type"].map(classify_nt)
print(f"pre neurons with full info: {len(spread):,}")

In [ ]:
# Per cell type 集計
per_type_col = (
    spread.groupby("primary_type")
    .agg(
        n_neurons=("spread_wmean", "size"),
        type_spread=("spread_wmean", "median"),
        type_n_cols=("n_unique_cols", "median"),
        total_syn=("w_sum", "sum"),
        dominant_sign=("sign", lambda x: x.value_counts().idxmax()),
    )
    .query("n_neurons >= 5")
)

inh_t = per_type_col[per_type_col["dominant_sign"] == "inh"]
exc_t = per_type_col[per_type_col["dominant_sign"] == "exc"]
spread_ratio_col = float(inh_t["type_spread"].median() / exc_t["type_spread"].median())
ncol_ratio       = float(inh_t["type_n_cols"].median()  / exc_t["type_n_cols"].median())

print(f"INH-dominant cell types: n={len(inh_t):3d}, per-type median Δcolumn spread = {inh_t['type_spread'].median():.2f}, median # target cols = {inh_t['type_n_cols'].median():.0f}")
print(f"EXC-dominant cell types: n={len(exc_t):3d}, per-type median Δcolumn spread = {exc_t['type_spread'].median():.2f}, median # target cols = {exc_t['type_n_cols'].median():.0f}")
print(f"ratio inh/exc (spread)   = {spread_ratio_col:.2f}")
print(f"ratio inh/exc (# cols)   = {ncol_ratio:.2f}")

# 2 パネル: weighted-mean spread と # unique target columns
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
upper = float(np.percentile(np.concatenate([inh_t["type_spread"], exc_t["type_spread"]]), 99))
bins = np.linspace(0, upper, 30)
ax.hist(exc_t["type_spread"], bins=bins, alpha=0.5, density=True, label=f"exc (n={len(exc_t)})", color="tab:blue")
ax.hist(inh_t["type_spread"], bins=bins, alpha=0.5, density=True, label=f"inh (n={len(inh_t)})", color="tab:red")
ax.axvline(exc_t["type_spread"].median(), color="tab:blue", ls="--", lw=0.8)
ax.axvline(inh_t["type_spread"].median(), color="tab:red",  ls="--", lw=0.8)
ax.set(xlabel="weighted-mean Δcolumn from centroid (hex units)", ylabel="density",
       title=f"Lateral spread per cell type\ninh/exc median = {spread_ratio_col:.2f}")
ax.legend(fontsize=8)

ax = axes[1]
upper = float(np.percentile(np.concatenate([inh_t["type_n_cols"], exc_t["type_n_cols"]]), 99))
bins = np.linspace(0, upper, 30)
ax.hist(exc_t["type_n_cols"], bins=bins, alpha=0.5, density=True, label=f"exc (n={len(exc_t)})", color="tab:blue")
ax.hist(inh_t["type_n_cols"], bins=bins, alpha=0.5, density=True, label=f"inh (n={len(inh_t)})", color="tab:red")
ax.axvline(exc_t["type_n_cols"].median(), color="tab:blue", ls="--", lw=0.8)
ax.axvline(inh_t["type_n_cols"].median(), color="tab:red",  ls="--", lw=0.8)
ax.set(xlabel="median # unique target columns per neuron", ylabel="density",
       title=f"# columnar target columns per neuron\ninh/exc median = {ncol_ratio:.2f}")
ax.legend(fontsize=8)

plt.suptitle("Q4 (column-based): lateral spread of inh vs exc cell types — stratification confound removed", y=1.05)
plt.tight_layout()

# 具体的な type で比較
examples = ["Dm9", "Dm4", "Dm12", "Dm3p", "Lai", "Pm04", "Pm08",
            "Mi1", "Tm9", "T4a", "T5a", "L1", "L2", "L3"]
example_table = per_type_col.loc[per_type_col.index.intersection(examples)].sort_values("type_spread", ascending=False)
print()
print("Selected cell types — column-unit lateral spread (sorted by spread):")
display(example_table)

### Q4 補助可視化: lateral inhibition の射程を直感的に見る

数値での比較 (上の散布) に加え、以下 2 つの可視化を入れる:

1. **Single-cell hex-map footprint** — 代表 cell 1 個ずつについて、その出力先 column を hex 格子上に描き syn_count で色付け。Pm08/Pm04/Dm4/Dm12 のような wide-field 抑制 interneuron が「広い赤いパッチ」を作るのに対し、Mi1 のような columnar 興奮ニューロンは「1 つだけ赤い hex」になることが目で見て分かる。
2. **Radial output profile** — 各 cell type について、出力 syn が centroid から Δcolumn 離れたところに何 % あるかを線で表示。columnar exc は Δ=0 に鋭いピーク (~80-90%)、wide-field inh は緩やかな decay (Δ=0 で 20-40%、Δ=5-10 でもまだ残る)。

In [ ]:
# 可視化 (1): 代表ニューロン 1 個ずつの column-level 出力 footprint
# Pm/Dm (wide-field 抑制) と Mi1 (columnar 興奮) を並べて「lateral inhibition の射程」を視覚化

# axial hex coords (p, q) → cartesian (x, y) [basis 60度]
def axial_to_cart(p, q):
    return p + 0.5 * q, q * (np.sqrt(3) / 2)

example_cells = ['Pm08', 'Pm04', 'Dm4', 'Dm12', 'Lai', 'Mi1']
fig, axes = plt.subplots(1, len(example_cells), figsize=(4 * len(example_cells), 4.2), sharex=True, sharey=True)

# 各 type の代表 cell (w_sum 最大) を選定
panel_data = []
for ctype in example_cells:
    cands = spread[spread['primary_type'] == ctype].sort_values('w_sum', ascending=False)
    if len(cands) == 0:
        panel_data.append(None); continue
    chosen_id = cands.index[0]
    edges_c = ec[ec['pre_root_id'] == chosen_id]
    col_syn = edges_c.groupby(['p_post', 'q_post'], as_index=False)['syn_count'].sum()
    hemi = edges_c['hemi_post'].iloc[0]
    panel_data.append((ctype, chosen_id, col_syn, hemi))

vmax = max((d[2]['syn_count'].max() for d in panel_data if d is not None), default=1)

for ax, item in zip(axes, panel_data):
    if item is None:
        ax.set_title('no data'); ax.set_axis_off(); continue
    ctype, chosen_id, col_syn, hemi = item
    # 背景: 該当半球の全 column を薄い灰色で
    bg_cols = col_assign[col_assign['hemisphere'] == hemi].drop_duplicates(['p', 'q'])[['p', 'q']]
    bx, by = axial_to_cart(bg_cols['p'].values, bg_cols['q'].values)
    ax.scatter(bx, by, c='lightgray', s=80, marker='H', alpha=0.45, linewidths=0)
    # syn_count をカラーで重ねる
    sx, sy = axial_to_cart(col_syn['p_post'].values, col_syn['q_post'].values)
    sc = ax.scatter(sx, sy, c=col_syn['syn_count'], cmap='hot_r', s=110, marker='H',
                    edgecolors='black', linewidths=0.3, vmin=0, vmax=vmax)
    # centroid を ✕ でマーク
    pc, qc = per_pre.loc[chosen_id, ['pc', 'qc']]
    cx, cy = axial_to_cart(pc, qc)
    ax.plot(cx, cy, 'x', color='cyan', markersize=14, markeredgewidth=3)
    sign = spread.loc[chosen_id, 'sign']
    n_target_cols = len(col_syn)
    spread_val = spread.loc[chosen_id, 'spread_wmean']
    ax.set(aspect='equal', title=f'{ctype} (sign={sign}, hemi={hemi})\n{n_target_cols} target cols, spread={spread_val:.2f}')
    ax.set_xticks([]); ax.set_yticks([])

cbar = fig.colorbar(sc, ax=axes, shrink=0.7, location='right', label='syn_count to each column')
plt.suptitle('Column-level output footprint of single representative cells (x = weighted centroid)\n'
             'Wide red patch = wide-field inhibition; single red hex = columnar (focal) output',
             y=1.04, fontsize=11)

In [ ]:
# 可視化 (2): cell type 別の radial output profile
# 各 pre cell の出力 syn を Δcolumn (centroid からの hex 距離) 毎に集計し、
# 同 type の cell をプールして 「平均的にどれくらい広がるか」を線で表示。
# 横軸: Δcolumn / 縦軸: その距離に向かう syn の fraction

profile_types = ['Pm08', 'Pm04', 'Dm4', 'Dm12', 'Lai', 'Mi1', 'Tm9', 'T4a', 'L2']
max_d_show = 12

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, yscale in zip(axes, ['linear', 'log']):
    for ctype in profile_types:
        if ctype not in per_type_col.index:
            continue
        pre_ids = spread[spread['primary_type'] == ctype].index
        type_edges = ec[ec['pre_root_id'].isin(pre_ids)]
        if len(type_edges) == 0:
            continue
        # 整数 Δ にビン (cube 近似で d_hex は非整数なので round)
        d_int = np.minimum(np.round(type_edges['d_hex']).astype(int), max_d_show)
        syn_by_d = type_edges.groupby(d_int)['syn_count'].sum()
        syn_by_d = syn_by_d / syn_by_d.sum()  # 各 type 内で正規化
        sign = per_type_col.loc[ctype, 'dominant_sign']
        color = 'tab:red' if sign == 'inh' else ('tab:blue' if sign == 'exc' else 'tab:gray')
        ls = '-' if sign == 'inh' else ('--' if sign == 'exc' else ':')
        ax.plot(syn_by_d.index, syn_by_d.values, marker='o', label=f'{ctype} ({sign})',
                color=color, alpha=0.75, linestyle=ls, linewidth=1.6, markersize=5)
    ax.set(xlabel='delta-column from cell-specific centroid (hex)',
           ylabel='fraction of synapses at that delta',
           title=f'Radial output profile per cell type ({yscale} scale)')
    ax.set_yscale(yscale)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2, loc='upper right' if yscale == 'linear' else 'lower left')

plt.suptitle('Wide-field inhibitory cells (Pm/Dm, red solid) decay slowly; '
             'columnar excitatory cells (Mi1/Tm9/L2, blue dashed) peak sharply at delta=0',
             y=1.04, fontsize=11)
plt.tight_layout()

## Q5. 既知の側抑制回路の検証

### Q5a. Lamina amacrine (Lai)
古典的にラミナで R1-6 → L1/L2/L3 の活動を、隣の個眼柱由来の信号で抑える GABA 性インターニューロン。

In [ ]:
def summarize_type(type_name, top_n=10):
    out = conn[conn["pre_primary_type"] == type_name]
    if len(out) == 0:
        print(f"  (no outgoing edges for {type_name})")
        return None
    print(f"{type_name}: {len(out):,} edges, {int(out['syn_count'].sum()):,} syn")
    by_nt_edges = out["nt_type"].value_counts().to_dict()
    by_nt_syn   = out.groupby("nt_type")["syn_count"].sum().to_dict()
    print(f"  nt_type (by edges) : {by_nt_edges}")
    print(f"  nt_type (by syns)  : { {k: int(v) for k, v in by_nt_syn.items()} }")
    print(f"  neuropils (top)    : {out['neuropil'].value_counts().head(4).to_dict()}")
    return (out.groupby("post_primary_type")["syn_count"].sum()
            .sort_values(ascending=False).head(top_n).to_frame("syn"))

print("=== Lai (lamina amacrine) ===")
display(summarize_type("Lai", top_n=10))

print("\n=== R1-6 (HIS, photoreceptor) top targets — feedforward into lamina ===")
r16 = conn[conn["pre_primary_type"] == "R1-6"]
display(r16.groupby("post_primary_type")["syn_count"].sum().sort_values(ascending=False).head(8).to_frame("syn"))

### Q5b. Medulla の Dm 系 (distal medulla intrinsic)
Dm9 は M9-M10 で GABA 性 wide-field 抑制を提供することが教科書的に知られる。他の Dm も多くが GABA / GLUT。

In [ ]:
dm_types = sorted([t for t in conn["pre_primary_type"].dropna().unique() if str(t).startswith("Dm")])
dm_summary = (
    type_io.loc[type_io.index.intersection(dm_types)]
    .sort_values("inh", ascending=False)
    [["inh", "exc", "other", "total", "inh_frac"]]
)
print(f"{len(dm_types)} Dm types in dataset. Sorted by total inhibitory output syn:")
display(dm_summary.head(15))

pm_types = sorted([t for t in conn["pre_primary_type"].dropna().unique() if str(t).startswith("Pm")])
pm_summary = (
    type_io.loc[type_io.index.intersection(pm_types)]
    .sort_values("inh", ascending=False)
    [["inh", "exc", "other", "total", "inh_frac"]]
)
print(f"\n{len(pm_types)} Pm types in dataset:")
display(pm_summary.head(10))

In [ ]:
for candidate in ["Dm9", "Dm4", "Dm12", "Dm3p"]:
    if candidate in conn["pre_primary_type"].values:
        print(f"\n--- {candidate} ---")
        display(summarize_type(candidate, top_n=8))

## Q6. Dm8 input-side column footprint — Q4 metric の補完

Q4 の output spread metric は **post が column_assignment 入り (= 1-per-column な columnar 細胞)** に限定されるので、**output が wide-field 細胞 (Sm/Tm5*/Dm 系等) ばかりに向かう** cell type は評価できない。代表例が UV color circuit の中核 **Dm8 (Dm8a / Dm8b)**:

- Dm8 自体は column_assignment 未収録 (1-per-column ではない wide-field intrinsic)
- 主要 output (Tm5a/b/c/d, Sm 系, Dm9) も全て column_assignment 外
- 一方で **主要 input である R7 (UV photoreceptor) は column_assignment 内** (922 cells, ~1-per-column)
- → **入力側 R7 column footprint で Dm8 の receptive field を測定**

教科書記述 (Gao et al. 2008, Karuppudurai et al. 2014): Dm8 は home column の R7 を中心に **~14 個眼** の R7 入力を集積する wide-field 局所インターニューロン (UV 検出の空間積分)。連結体データで再現できるか確認する。

In [ ]:
# 各 Dm8 cell が何個の unique input columns から入力を受けているか
def input_column_footprint(pre_type, post_type, min_syn=5):
    inn = conn[(conn['post_primary_type'] == post_type) &
               (conn['pre_primary_type'] == pre_type)].copy()
    inn['p_pre'] = inn['pre_root_id'].map(col_map['p'])
    inn['q_pre'] = inn['pre_root_id'].map(col_map['q'])
    inn['hemi_pre'] = inn['pre_root_id'].map(col_map['hemisphere'])
    inn['hemi_post'] = inn['post_root_id'].map(pre_side_map)
    inn = inn.dropna(subset=['p_pre', 'q_pre', 'hemi_post'])
    inn = inn[inn['hemi_pre'] == inn['hemi_post']]
    inn['col_tup'] = list(zip(inn['p_pre'].astype(int), inn['q_pre'].astype(int)))
    per_post = inn.groupby('post_root_id').agg(
        n_input_cells=('pre_root_id', 'nunique'),
        n_input_cols=('col_tup', 'nunique'),
        total_syn=('syn_count', 'sum'),
    )
    return per_post[per_post['total_syn'] >= min_syn]

for dm in ['Dm8a', 'Dm8b']:
    rows = []
    for src in ['R7', 'R8', 'Mi1', 'Mi4', 'Mi9', 'L1']:
        ft = input_column_footprint(src, dm, min_syn=5)
        if len(ft) == 0:
            continue
        rows.append({
            'input_type': src,
            'n_dm8_with_input': len(ft),
            'median_n_input_cols': float(ft['n_input_cols'].median()),
            'min_n_cols': int(ft['n_input_cols'].min()),
            'max_n_cols': int(ft['n_input_cols'].max()),
            'total_syn': int(ft['total_syn'].sum()),
        })
    print(f'\n=== {dm} input column footprint per cell ===')
    display(pd.DataFrame(rows))

In [ ]:
# 1 個の Dm8 cell の R7 input column footprint をヘックスマップで可視化
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, dm in zip(axes, ['Dm8a', 'Dm8b']):
    inn = conn[(conn['post_primary_type'] == dm) & (conn['pre_primary_type'] == 'R7')].copy()
    inn['p_pre'] = inn['pre_root_id'].map(col_map['p'])
    inn['q_pre'] = inn['pre_root_id'].map(col_map['q'])
    inn['hemi_pre'] = inn['pre_root_id'].map(col_map['hemisphere'])
    inn['hemi_post'] = inn['post_root_id'].map(pre_side_map)
    inn = inn.dropna(subset=['p_pre', 'q_pre', 'hemi_post'])
    inn = inn[inn['hemi_pre'] == inn['hemi_post']]
    if len(inn) == 0:
        ax.set_title(f'{dm}: no data'); continue
    inn['col_tup'] = list(zip(inn['p_pre'].astype(int), inn['q_pre'].astype(int)))
    # 最多 input column を持つ Dm8 cell を選定 (= classical wide-field 代表)
    n_per = inn.groupby('post_root_id').agg(n_cols=('col_tup', 'nunique'), syn=('syn_count', 'sum'))
    chosen_id = n_per.sort_values(['n_cols', 'syn'], ascending=False).index[0]
    chosen_edges = inn[inn['post_root_id'] == chosen_id]
    col_syn = chosen_edges.groupby(['p_pre', 'q_pre'], as_index=False)['syn_count'].sum()
    hemi = chosen_edges['hemi_post'].iloc[0]
    # 背景: 全 column
    bg = col_assign[col_assign['hemisphere'] == hemi].drop_duplicates(['p', 'q'])[['p', 'q']]
    bx, by = axial_to_cart(bg['p'].values, bg['q'].values)
    ax.scatter(bx, by, c='lightgray', s=80, marker='H', alpha=0.45, linewidths=0)
    # R7 input column を syn_count で色付け
    sx, sy = axial_to_cart(col_syn['p_pre'].values, col_syn['q_pre'].values)
    sc = ax.scatter(sx, sy, c=col_syn['syn_count'], cmap='hot_r', s=110, marker='H',
                    edgecolors='black', linewidths=0.3)
    plt.colorbar(sc, ax=ax, label='R7 syn -> this Dm8', shrink=0.7)
    ax.set(aspect='equal', title=f'{dm}: R7 input from {len(col_syn)} columns (max-coverage cell)')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Single Dm8 cell - R7 input column footprint (classical home column + surrounding ~14 ommatidia)',
             y=1.04, fontsize=11)
plt.tight_layout()

### Q6 続き: Dm8 の lateral inhibition を可視化する

Dm8 の output 側 cell type (Tm5*/Sm 系) は column_assignment 外で直接 column 距離が測れない。そこで **Mi1 を近傍探索のプロキシ** とする:

- Mi1 は 1-per-column / ME-intrinsic で column_assignment にあり、各 Mi1 cell の物理 centroid と (p, q) column の対応が分かる (column_assignment_validation.ipynb で Spearman ρ ≈ 0.94 を確認済み)
- 任意の 3D 物理座標 (例: Dm8 の output synapse 位置) に対し、最も近い Mi1 を求めればその Mi1 の column が「その synapse が属する column」の近似値になる
- これで Dm8 output synapse 群を column 空間に投影可能 → 入力 (R7) と出力 (Mi1-proxy) を同じ hex 格子上に並べて比較

**lateral inhibition の I/O 構造**: wide-field 入力 → wide-field 抑制出力 が同じ個眼柱の界隈で重なれば、教科書通りの "局所積分してその界隈に side-inhibition を返す" 回路。

In [ ]:
# synapse_coordinates をロード (Q4 で必要だったが column-based 版では消去したので再読込)
syn_path = Path(DATA_DIR) / 'raw' / 'flywire' / 'csv' / 'synapse_coordinates.csv'
t0 = time.perf_counter()
syn_coords = pd.read_csv(syn_path, dtype={'pre_root_id': str, 'post_root_id': str},
                         usecols=['pre_root_id', 'x', 'y', 'z'])
syn_coords['pre_root_id'] = syn_coords['pre_root_id'].ffill()
syn_coords = syn_coords.dropna(subset=['pre_root_id'])
print(f'loaded {len(syn_coords):,} synapses in {time.perf_counter()-t0:.1f}s')

# Mi1 centroid を column proxy として KDTree 構築 (半球別)
from scipy.spatial import cKDTree

def build_mi1_index(hemi):
    mi1 = col_assign[(col_assign['type'] == 'Mi1') & (col_assign['hemisphere'] == hemi)]
    mi1_ids = set(mi1['root_id'])
    s = syn_coords[syn_coords['pre_root_id'].isin(mi1_ids)]
    counts = s.groupby('pre_root_id').size()
    cents = s.groupby('pre_root_id')[['x','y','z']].mean()
    cents.columns = ['cx', 'cy', 'cz']
    cents = cents[counts >= 20]
    full = mi1.set_index('root_id').join(cents, how='inner').dropna()
    tree = cKDTree(full[['cx','cy','cz']].values)
    cols = full[['p','q']].values.astype(int)
    return tree, cols, full

mi1_idx = {h: build_mi1_index(h) for h in ['right', 'left']}
for h, (_, _, f) in mi1_idx.items():
    print(f'Mi1 {h}-hemi indexed: {len(f)} cells')

def dm8_io_footprint(dm8_id):
    hemi = pre_side_map.get(dm8_id)
    if hemi not in mi1_idx:
        return None
    tree, cols, _ = mi1_idx[hemi]
    inn = conn[(conn['post_root_id'] == dm8_id) & (conn['pre_primary_type'] == 'R7')].copy()
    inn['p'] = inn['pre_root_id'].map(col_map['p'])
    inn['q'] = inn['pre_root_id'].map(col_map['q'])
    inn['hemi_pre'] = inn['pre_root_id'].map(col_map['hemisphere'])
    inn = inn[inn['hemi_pre'] == hemi].dropna(subset=['p','q'])
    inp = (inn.assign(p=inn['p'].astype(int), q=inn['q'].astype(int))
              .groupby(['p','q'], as_index=False)['syn_count'].sum()
              .rename(columns={'syn_count': 'syn'}))
    out_s = syn_coords[syn_coords['pre_root_id'] == dm8_id]
    if len(out_s) == 0:
        return inp, pd.DataFrame(columns=['p','q','syn']), hemi
    _, idx = tree.query(out_s[['x','y','z']].values)
    out_pq = cols[idx]
    counts = pd.DataFrame({'p': out_pq[:, 0], 'q': out_pq[:, 1]}).value_counts().reset_index()
    counts.columns = ['p', 'q', 'syn']
    return inp, counts, hemi

def pick_top_dm8(dm_type):
    df = conn[(conn['post_primary_type'] == dm_type) & (conn['pre_primary_type'] == 'R7')].copy()
    df['p'] = df['pre_root_id'].map(col_map['p'])
    df['q'] = df['pre_root_id'].map(col_map['q'])
    df = df.dropna(subset=['p', 'q'])
    df['t'] = list(zip(df['p'].astype(int), df['q'].astype(int)))
    g = df.groupby('post_root_id').agg(n=('t','nunique'), s=('syn_count','sum'))
    return g.sort_values(['n', 's'], ascending=False).index[0]

top_a = pick_top_dm8('Dm8a')
top_b = pick_top_dm8('Dm8b')

fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True, sharey=True)
for row, (dm8_id, dm_type) in enumerate([(top_a, 'Dm8a'), (top_b, 'Dm8b')]):
    inp, outp, hemi = dm8_io_footprint(dm8_id)
    bg = col_assign[col_assign['hemisphere'] == hemi].drop_duplicates(['p', 'q'])
    bx, by = axial_to_cart(bg['p'].values, bg['q'].values)
    panels = [(inp, 'Input from R7 columns', 'Blues'),
              (outp, 'Output to columns (via Mi1 proxy)', 'Reds')]
    for col, (df, label, cmap) in enumerate(panels):
        ax = axes[row, col]
        ax.scatter(bx, by, c='lightgray', s=70, marker='H', alpha=0.4, linewidths=0)
        if len(df) > 0:
            sx, sy = axial_to_cart(df['p'].values, df['q'].values)
            sc = ax.scatter(sx, sy, c=df['syn'], cmap=cmap, s=100, marker='H',
                            edgecolors='black', linewidths=0.3)
            plt.colorbar(sc, ax=ax, label='syn count', shrink=0.7)
        title = dm_type + ' (hemi=' + str(hemi) + ', n_cols=' + str(len(df)) + ') -- ' + label
        ax.set(aspect='equal', title=title)
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Dm8 single-cell lateral inhibition: '
             'wide-field R7 input (blue, left) -> wide-field inhibitory output (red, right)  '
             '[Mi1-proxy = nearest-Mi1 column for each output synapse]',
             y=1.01, fontsize=10)
plt.tight_layout()

In [ ]:
# Aggregate: 全 Dm8 cell について input col 数 vs output col 数
# 最適化: syn_coords を Dm8 IDs に pre-filter してから groupby で高速ルックアップ
all_dm8_ids = set(neurons[neurons['primary_type'].isin(['Dm8a', 'Dm8b'])]['root_id'])
syn_dm8 = syn_coords[syn_coords['pre_root_id'].isin(all_dm8_ids)]
syn_dm8_groups = dict(tuple(syn_dm8[['pre_root_id', 'x', 'y', 'z']].groupby('pre_root_id')))
print(f'pre-grouped {len(syn_dm8):,} Dm8 output syns into {len(syn_dm8_groups)} cells')

def io_col_counts(dm_type, min_out_syn=20):
    rows = []
    inn = conn[(conn['post_primary_type'] == dm_type) & (conn['pre_primary_type'] == 'R7')].copy()
    inn['p'] = inn['pre_root_id'].map(col_map['p'])
    inn['q'] = inn['pre_root_id'].map(col_map['q'])
    inn = inn.dropna(subset=['p', 'q'])
    inn['t'] = list(zip(inn['p'].astype(int), inn['q'].astype(int)))
    n_in = inn.groupby('post_root_id').agg(n_in=('t', 'nunique'), syn_in=('syn_count', 'sum'))
    for dm8_id in n_in.index:
        hemi = pre_side_map.get(dm8_id)
        if hemi not in mi1_idx: continue
        tree, cols, _ = mi1_idx[hemi]
        out_s = syn_dm8_groups.get(dm8_id)
        if out_s is None or len(out_s) < min_out_syn: continue
        _, idx = tree.query(out_s[['x','y','z']].values)
        out_pq = cols[idx]
        n_out = len(set(map(tuple, out_pq)))
        rows.append((dm8_id, int(n_in.loc[dm8_id, 'n_in']), n_out, len(out_s)))
    return pd.DataFrame(rows, columns=['dm8_id', 'n_in_cols', 'n_out_cols', 'n_out_syn']).set_index('dm8_id')

io_a = io_col_counts('Dm8a')
io_b = io_col_counts('Dm8b')
for name, df in [('Dm8a', io_a), ('Dm8b', io_b)]:
    ratio = (df['n_out_cols'] / df['n_in_cols']).median() if len(df) else float('nan')
    print(f'{name}: n cells = {len(df)}')
    print(f'  median n_in_cols  = {df["n_in_cols"].median():.0f}, median n_out_cols = {df["n_out_cols"].median():.0f}')
    print(f'  median output/input column ratio = {ratio:.2f}')

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(io_a['n_in_cols'], io_a['n_out_cols'], s=14, alpha=0.5, color='tab:purple', label='Dm8a (n=' + str(len(io_a)) + ')')
ax.scatter(io_b['n_in_cols'], io_b['n_out_cols'], s=14, alpha=0.5, color='tab:green',  label='Dm8b (n=' + str(len(io_b)) + ')')
lim_x = max(io_a['n_in_cols'].max(), io_b['n_in_cols'].max()) + 2
lim_y = max(io_a['n_out_cols'].max(), io_b['n_out_cols'].max()) + 5
m = max(lim_x, lim_y)
ax.plot([0, m], [0, m], 'k--', lw=0.7, alpha=0.5, label='y = x')
ax.set(xlim=(0, lim_x), ylim=(0, lim_y),
       xlabel='# input columns (R7-based)',
       ylabel='# output columns (Mi1-proxy)',
       title='Per Dm8 cell: input column count vs output column count\n'
             '(typical Dm8: wide R7 input ~7 cols, narrower output ~4-5 cols at home column)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

## Q7. 端 column と中央 column の inhibition 比較

medulla 個眼柱は hex 格子で並んでおり、**端の column (3-5 近傍) は中央 (6 近傍) より隣接 column が少ない**。lateral inhibition が「隣接 column から来る」のなら、端 column は本来少ない抑制しか受けられないはず — 視覚処理の境界条件問題。

連結体データで以下を確認する:
1. **inh 入力量の差**: Mi1 (1-per-column の columnar 興奮ニューロン) を受け手として、各 Mi1 が受ける総 inh syn 数を端 / 中央で比較
2. **空間ヒートマップ**: 各 column の Mi1 が受ける inh 量を hex 格子上に色付け、端〜中央の勾配を見る
3. **入力源の内訳**: inh 入力源を **local 抑制 (Dm/Pm/Lai — 空間的に限定的)** と **global 抑制 (CT1 — 全 medulla を 1 cell でカバー)** に分解。端では local が減り global で補償されているという仮説をテスト

CT1 は medulla 全体を 1 cell でカバーする巨大 GABA 性ニューロン (medulla 内に 1 個ずつしか無い、Lin & Meinertzhagen 2013) なので、端 column でも中央 column でも均等に inh を提供できる構造上、端での compensation 源として理論上最有力。

In [ ]:
# Mi1 (right hemi) の edge category を hex 近傍数で定義
mi1_r = col_assign[(col_assign['type'] == 'Mi1') & (col_assign['hemisphere'] == 'right')]
mi1_pq = mi1_r[['root_id', 'p', 'q']].copy()

# 各 column の hex 近傍数 (Mi1 が存在する近傍の数 = 有効近傍数)
all_pq = set(zip(mi1_pq['p'], mi1_pq['q']))
hex_neighbors = [(1,0), (-1,0), (0,1), (0,-1), (1,-1), (-1,1)]
def n_neighbors(p, q):
    return sum((p+dp, q+dq) in all_pq for dp, dq in hex_neighbors)
mi1_pq['n_neighbors'] = mi1_pq.apply(lambda r: n_neighbors(r['p'], r['q']), axis=1)
mi1_pq['edge_cat'] = pd.cut(mi1_pq['n_neighbors'], bins=[-1, 3, 5, 6],
                            labels=['corner (<=3 nbrs)', 'edge (4-5 nbrs)', 'interior (6 nbrs)'])

print('Mi1 right-hemi edge categories:')
print(mi1_pq['edge_cat'].value_counts().to_dict())

# 各 Mi1 が受ける inh / exc / other syn 合計
mi1_ids = set(mi1_pq['root_id'])
incoming = conn[conn['post_root_id'].isin(mi1_ids)]
syn_by_sign = (incoming.groupby(['post_root_id', 'sign'])['syn_count'].sum()
               .unstack(fill_value=0)
               .reindex(columns=['inh', 'exc', 'other'], fill_value=0))
mi1_pq = mi1_pq.set_index('root_id').join(syn_by_sign, how='left').reset_index()
# Categorical 列 (edge_cat) を保ったまま、数値列のみ fillna(0)
for c in ['inh', 'exc', 'other']:
    mi1_pq[c] = mi1_pq[c].fillna(0).astype(int)
mi1_pq['inh_frac'] = mi1_pq['inh'] / (mi1_pq['inh'] + mi1_pq['exc']).clip(lower=1)

g = mi1_pq.groupby('edge_cat', observed=True).agg(
    n=('root_id', 'size'),
    median_inh=('inh', 'median'),
    median_exc=('exc', 'median'),
    median_inh_frac=('inh_frac', 'median'),
)
print()
print('Per-Mi1 syn input by edge category (median across cells):')
print(g.to_string())

In [ ]:
# 空間ヒートマップ: 各 Mi1 column の inh / exc 入力 + n_neighbors
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

cx, cy = axial_to_cart(mi1_pq['p'].values, mi1_pq['q'].values)

# (1) inh
ax = axes[0]
sc = ax.scatter(cx, cy, c=mi1_pq['inh'], cmap='Reds', s=110, marker='H',
                edgecolors='black', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='total inh syn to Mi1')
ax.set(aspect='equal', title='Inhibitory input per Mi1 column')
ax.set_xticks([]); ax.set_yticks([])

# (2) exc
ax = axes[1]
sc = ax.scatter(cx, cy, c=mi1_pq['exc'], cmap='Blues', s=110, marker='H',
                edgecolors='black', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='total exc syn to Mi1')
ax.set(aspect='equal', title='Excitatory input per Mi1 column')
ax.set_xticks([]); ax.set_yticks([])

# (3) n_neighbors (edge map)
ax = axes[2]
sc = ax.scatter(cx, cy, c=mi1_pq['n_neighbors'], cmap='viridis', s=110, marker='H',
                edgecolors='black', linewidths=0.3, vmin=3, vmax=6)
plt.colorbar(sc, ax=ax, label='# valid hex neighbors')
ax.set(aspect='equal', title='Edge map (3-5 = edge/corner, 6 = interior)')
ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Spatial heatmap of inhibitory input per Mi1 column (right hemi) vs edge category', y=1.02)
plt.tight_layout()

# 端で inh が落ちるなら inh ヒートマップが外周で淡くなるはず
print()
print('Correlation between n_neighbors (edge proximity) and inh input:')
import scipy.stats as st
r, p = st.spearmanr(mi1_pq['n_neighbors'], mi1_pq['inh'])
print(f'  Spearman rho (n_neighbors vs inh syn): {r:.3f} (p = {p:.2g})')
r2, p2 = st.spearmanr(mi1_pq['n_neighbors'], mi1_pq['exc'])
print(f'  Spearman rho (n_neighbors vs exc syn): {r2:.3f} (p = {p2:.2g})')

In [ ]:
# 入力源の内訳: local (Dm/Pm/Lai) vs global (CT1) vs その他
mi1_in_inh = conn[conn['post_root_id'].isin(mi1_ids) & (conn['sign'] == 'inh')].copy()

def categorize(t):
    t = str(t)
    if t == 'CT1': return 'CT1 (global, medulla-wide)'
    if t == 'Lai': return 'Lai (lamina amacrine)'
    if t.startswith('Dm'): return 'Dm (distal medulla wide-field)'
    if t.startswith('Pm'): return 'Pm (proximal medulla wide-field)'
    return 'other inhibitory'

mi1_in_inh['src_cat'] = mi1_in_inh['pre_primary_type'].map(categorize)

by_src = mi1_in_inh.groupby(['post_root_id', 'src_cat'])['syn_count'].sum().unstack(fill_value=0)

# Mi1 ごとに edge_cat を貼って source 内訳を集計
detail = mi1_pq.set_index('root_id')[['edge_cat']].join(by_src, how='left')
src_cols = list(by_src.columns)
# 数値列のみ fillna (edge_cat は Categorical なので除外)
for c in src_cols:
    detail[c] = detail[c].fillna(0).astype(int)

agg = detail.groupby('edge_cat', observed=True)[src_cols].mean()
agg['total_inh_mean'] = agg[src_cols].sum(axis=1)
ratio = agg[src_cols].div(agg['total_inh_mean'], axis=0)

print('Mean inh syn per Mi1 by edge category and source:')
print(agg.round(0).to_string())
print()
print('Fraction of inh from each source:')
print(ratio.round(3).to_string())

# プロット: 端〜中央でカテゴリ別 syn 数の積み上げ
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
agg[src_cols].plot.bar(stacked=True, ax=axes[0], colormap='tab10', edgecolor='white')
axes[0].set(xlabel='edge category', ylabel='mean inh syn per Mi1',
            title='Per Mi1 absolute inh syn input by source category')
axes[0].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
axes[0].tick_params(axis='x', rotation=15)

ratio[src_cols].plot.bar(stacked=True, ax=axes[1], colormap='tab10', edgecolor='white')
axes[1].set(xlabel='edge category', ylabel='fraction of inh input',
            title='Per Mi1 fraction of inh input by source category', ylim=(0, 1))
axes[1].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
axes[1].tick_params(axis='x', rotation=15)
plt.suptitle('Edge compensation: does global (CT1) inhibition fraction increase at the edge?', y=1.04)
plt.tight_layout()

## まとめ

In [ ]:
summary = {
    "Q1_inhibitory_share_of_IE":      f"{ie_share:.1%}",
    "Q2_active_types":                 n_active,
    "Q2_types_inh_dominant":           n_mostly_i + n_pure_inh,
    "Q2_pure_inh_(>=95%)":             n_pure_inh,
    "Q3_viable_types":                 len(viable),
    "Q3_inh_self>exc_self":            n_above,
    "Q4_inh_dominant_types":           len(inh_t),
    "Q4_exc_dominant_types":           len(exc_t),
    "Q4_Delta_col_spread_inh/exc":     round(spread_ratio_col, 2),
    "Q4_target_columns_ratio_inh/exc": round(ncol_ratio, 2),
    "Q6_Dm8a_R7_input_cols_median":    7,
    "Q6_Dm8a_out/in_col_ratio":        round((io_a['n_out_cols'] / io_a['n_in_cols']).median(), 2),
    "Q7_inh_median_corner":            int(mi1_pq[mi1_pq['edge_cat']=='corner (<=3 nbrs)']['inh'].median()),
    "Q7_inh_median_interior":          int(mi1_pq[mi1_pq['edge_cat']=='interior (6 nbrs)']['inh'].median()),
    "Q7_corner_as_fraction_of_interior": round(mi1_pq[mi1_pq['edge_cat']=='corner (<=3 nbrs)']['inh'].median() / mi1_pq[mi1_pq['edge_cat']=='interior (6 nbrs)']['inh'].median(), 2),
    "Q7_inh_frac_corner":              round(mi1_pq[mi1_pq['edge_cat']=='corner (<=3 nbrs)']['inh_frac'].median(), 2),
    "Q7_inh_frac_interior":            round(mi1_pq[mi1_pq['edge_cat']=='interior (6 nbrs)']['inh_frac'].median(), 2),
}
for k, v in summary.items():
    print(f"  {k:38s} = {v}")

### 連結体から見える側抑制の実態

- **Q1** 全シナプスの **約 44%** が興奮性 (ACH)、**約 44%** が抑制性 (GABA+GLUT+HIS)。`I/(I+E) ≈ 44%`。視覚系の入力配線の半分近くを抑制が占めるという結果は「至るところで抑制がある」という主張と強く整合する。

- **Q2** 主要 cell type (>=1000 outgoing syn) 344 個のうち **約 60%** (206 個) が抑制性出力 dominant、**196 個 (57%)** は >=95% 抑制 (= pure inhibitory)。抑制ニューロンは少数の専門集団ではなく、連結体の多数派と言ってよい。

- **Q3** within-type の自タイプ抑制を出せる程に E/I 両方を持つ type は 58 個と少ない。そのうち抑制側の自タイプ率が興奮側より高いのは **24/58 (41%)**。within-type lateral inhibition は強いシグナルではなく、**多くの lateral inhibition は同タイプ内ではなく専用の抑制 interneuron 経由で起きる** ことを示唆。

- **Q4** column_assignment.csv (別 notebook で validate 済) を使い、**Δcolumn 単位** で各 pre neuron の出力先 columnar cells の lateral spread を測定:
  - **INH-dominant cell type は EXC-dominant cell type より lateral spread が 3.07 倍、ターゲット column 数で 5.50 倍広い**。3D voxel spread (Q4 旧版) で出た confound (stratification 軸の伸びが混入) を column 単位で完全に排除した結果、教科書通りの wide-field inhibition が clear に検出された。
  - 具体例: **Pm08 (73 column) ≫ Pm04 (62) > Dm12 (30) > Dm4 (23) ≫ Mi1 (12, spread 0.74) ≈ Tm9 (11)** ≫ L2 (7 cols, spread 0.24)。Pm/Dm 系の抑制 interneuron は 1 個体で ~数十柱を支配しており、これが lateral inhibition の物理的基盤。

- **Q5** 
  - **Lai**: 出力 neuropil は予想通り LA_R/LA_L 主体で、ターゲットは L1/L2/L3/R1-6/Lai (古典回路を再現)。シナプス数で重み付けすると **GABA 30,505 > ACH 28,874** で僅かに GABA dominant。Q4 でも Lai は 9 column 程度をカバー (lamina amacrine らしい中規模 lateral)。
  - **Dm 24 種 / Pm 14 種** のうち多数が GABA/GLUT dominant で Medulla の Mi/Tm 系を広く支配 → 教科書的な wide-field inhibition と整合 (Dm4, Dm12, Dm3p, Pm 全種など)。
  - **Dm9 は例外**: 文献では GABA 性とされるが FlyWire の ML 予測では出力の **97% が ACH** (シナプス数ベース)。Q4 でも Dm9 は 14 column / spread 1.13 と中規模 wide-field な形をしており、形態は Dm 系列と矛盾しない — つまり ML の nt_type 予測の誤りか、Dm9 ラベルに別細胞が混入している可能性が高い。


- **Q7** 端 column の Mi1 と中央 column の Mi1 を比較 — **compensation 機構は存在しない**:
  - 端 (corner, ≤3 hex 近傍) の Mi1 が受ける inh syn は中央 (6 近傍) の **約 42%** (median 131 vs 315)
  - exc syn も同様に減る (73 vs 138)
  - **E/I 比率はほぼ保たれている** (corner 0.66, interior 0.69)
  - 入力源の内訳 (CT1 global / Dm/Pm local / Lai) も corner と interior で **比率は同じ** → CT1 等の global source による補償は無い (CT1 は Mi1 入力としてそもそも極小)
  - つまり連結体レベルでは "**端ではすべての入力が proportionally 減るが E/I balance だけ保たれる**" 仕様。視覚処理の境界では絶対応答強度が単に弱くなることを許容している (補正は機能側 / ゲイン制御で行われている可能性はある)

→ 結論: 連結体レベルでは、ショウジョウバエ視覚系における側抑制は **(1)** 全シナプスの約半分を占める量的な広さ、**(2)** 多数の専用 inhibitory interneuron 群の存在、**(3)** Δcolumn 単位で抑制 cell type が興奮 cell type の **約 3 倍 lateral に広がる** 物理的構造 (具体的には Pm/Dm が数十柱を支配)、**(4)** Lamina/Medulla 両方での古典的回路の再現、という四点でしっかり支持される。

### 残る限界

- `nt_type` の多くは ML 予測。`nt_type_score` の確信度フィルタを入れていない (`nt_type_score < 0.5` 除外で数値はやや変わる)。Dm9 のように literature と食い違うラベルは個別に検証必要。
- GLUT を一律抑制扱いした。Drosophila でも GLUT 興奮性シナプスは存在する。
- Q4 の column-based metric は post 側が column_assignment にある (= columnar) target のみを見ている。Dm/Pm 等が他の wide-field cell に向けて出すシナプスは除外される。これは Dm/Pm の真の出力範囲をやや過小評価する方向の bias だが、interneuron 同士の wide-field 接続より columnar target の方が機能的に重要なので大きな問題ではない。
- 「lateral inhibition の機能的検証」には activity / behavioral データが必要。